# 응용 02 — Confidence, NMS, 정답 매칭을 처음부터 이해하기

`02_nms.py`의 흐름을 작은 표와 숫자로 풀어보는 노트북입니다. 셀은 위에서 아래로 **Shift+Enter**로 실행하세요. 변수는 같은 커널 안에서 다음 셀로 이어집니다. 중간부터 실행해 변수가 없다는 메시지가 나오면 첫 셀부터 실행하세요.

이 자료는 개념과 계산 과정을 설명합니다. 원본 구현의 검증 여부, 옳고 그름, 제출용 판단은 직접 기록할 수 있도록 남겨둡니다. 원본 `.py`와 기존 CSV는 수정하지 않습니다.

준비 패키지: `numpy`, `pandas`, `torch`, `torchvision`. 설치가 필요한 환경에서는 별도 셀에서 `%pip install numpy pandas torch torchvision`을 실행하고 커널을 재시작하세요.

학습 순서: CSV → Tensor → confidence 필터 → NMS → 원본 인덱스 → 정답 매칭 → 지표.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

roots = [Path.cwd(), *Path.cwd().parents]
BASE = next((root / "20261106/02_advanced" for root in roots
             if (root / "20261106/02_advanced/data/manifest.csv").is_file()), None)
if BASE is None:
    raise FileNotFoundError("저장소 안에서 노트북을 실행하세요.")
DATA = BASE / "data"
# 학습 예제 파일은 기존 실습 결과와 구분해 저장합니다.
OUT = BASE / "outputs/notebook_examples"
print("입력 폴더:", DATA)
import torch
from torchvision.ops import box_iou, batched_nms
p = pd.read_csv(DATA / "boxes_fixture.csv")
g = pd.read_csv(DATA / "boxes_gt.csv")
print("예측\n" + p.to_string())
print("\n정답\n" + g.to_string())


입력 폴더: /home/swchoi/projects/python/practice_assignment/20261106/02_advanced/data
예측
  image_id  class_id  confidence  x1  y1   x2  y2
0    scene         0        0.95  14  20   52  58
1    scene         0        0.85  16  20   54  58
2    scene         1        0.80  70  60  108  98
3    scene         2        0.40  90   6  110  22

정답
  image_id  class_id   x1  y1   x2  y2
0    scene         0   12  18   52  58
1    scene         1   70  60  108  98
2    scene         2  116  18  150  62


## 1. 예측 4개, 정답 3개

예측의 각 행은 모델이 찾았다고 보고한 객체 하나입니다. `class_id`는 종류 번호이고 `confidence`는 모델 점수입니다. `(x1,y1)`은 왼쪽 위, `(x2,y2)`는 오른쪽 아래 좌표입니다. `image_id`는 어느 이미지인지 나타냅니다. 이 실습 데이터의 이미지는 `scene` 하나입니다.

`p[["x1", "y1", "x2", "y2"]]`는 네 열을 순서대로 고릅니다. `.to_numpy()`는 표를 숫자 배열로 바꾸고 `torch.tensor()`는 PyTorch가 계산할 Tensor로 바꿉니다. `float32`는 소수 좌표·점수, `int64`는 클래스 번호에 사용합니다. `boxes`의 모양 `(4,4)`는 **박스 4개 × 좌표 4개**라는 뜻입니다.

In [2]:
boxes = torch.tensor(p[["x1", "y1", "x2", "y2"]].to_numpy(), dtype=torch.float32)
scores = torch.tensor(p["confidence"].to_numpy(), dtype=torch.float32)
classes = torch.tensor(p["class_id"].to_numpy(), dtype=torch.int64)
gt_boxes = torch.tensor(g[["x1", "y1", "x2", "y2"]].to_numpy(), dtype=torch.float32)
print("boxes:", boxes)
print("shape:", boxes.shape)
print("scores:", scores)
print("classes:", classes)


boxes: tensor([[ 14.,  20.,  52.,  58.],
        [ 16.,  20.,  54.,  58.],
        [ 70.,  60., 108.,  98.],
        [ 90.,   6., 110.,  22.]])
shape: torch.Size([4, 4])
scores: tensor([0.9500, 0.8500, 0.8000, 0.4000])
classes: tensor([0, 0, 1, 2])


## 2. confidence 필터: 어떤 예측을 평가 대상으로 남길까요?

`scores >= conf`는 각 점수가 기준 이상인지 나타내는 True/False 배열입니다. `torch.where(...)[0]`는 True인 위치 번호만 꺼냅니다. 이 위치 번호가 `ids`입니다. `boxes[ids]`는 해당 위치의 박스만 모은 새 배열입니다.

예를 들어 conf=0.50이면 원본 0·1·2번을 남기고 0.40인 3번은 제외합니다. 여기서 **confidence 기준은 정답과 얼마나 겹치는지에 대한 기준이 아닙니다.**

In [3]:
for conf in [0.25, 0.50]:
    mask = scores >= conf
    ids = torch.where(mask)[0]
    print("conf:", conf, "조건:", mask.tolist(), "원본 인덱스:", ids.tolist())


conf: 0.25 조건: [True, True, True, True] 원본 인덱스: [0, 1, 2, 3]
conf: 0.5 조건: [True, True, True, False] 원본 인덱스: [0, 1, 2]


## 3. IoU: 두 박스가 얼마나 겹칠까요?

**IoU = 교집합 넓이 / 합집합 넓이**입니다. 0은 겹침 없음, 1은 완전 일치입니다. `box_iou(A,B)`는 A의 각 박스와 B의 각 박스 사이의 IoU를 계산해 N×M 표를 만듭니다.

다음 첫 표는 예측끼리, 두 번째 표는 예측과 정답 사이의 겹침입니다. NMS는 예측끼리의 IoU를 보고, 정답 매칭은 예측과 정답 사이의 IoU를 봅니다. 두 단계는 비교 대상부터 다릅니다.

In [4]:
prediction_iou = box_iou(boxes, boxes)
gt_iou = box_iou(boxes, gt_boxes)
print("예측끼리 IoU:\n", pd.DataFrame(prediction_iou.numpy()).round(6).to_string())
print("\n예측-정답 IoU:\n", pd.DataFrame(gt_iou.numpy(), columns=["GT0", "GT1", "GT2"]).round(6).to_string())


예측끼리 IoU:
      0    1    2    3
0  1.0  0.9  0.0  0.0
1  0.9  1.0  0.0  0.0
2  0.0  0.0  1.0  0.0
3  0.0  0.0  0.0  1.0

예측-정답 IoU:
         GT0  GT1  GT2
0  0.902500  0.0  0.0
1  0.816229  0.0  0.0
2  0.000000  1.0  0.0
3  0.000000  0.0  0.0


## 4. NMS: 같은 물체를 가리키는 겹친 예측 줄이기

NMS(Non-Maximum Suppression)는 점수가 높은 예측을 먼저 남기고, 그 예측과 많이 겹치는 같은 클래스 예측을 억제하는 방식입니다. NMS 단계는 정답 라벨을 사용하지 않습니다.

`batched_nms(boxes, scores, classes, iou_threshold)`는 클래스 번호가 같은 박스끼리만 억제합니다. IoU가 지정 기준보다 큰 박스를 억제하므로 NMS 기준을 높이면 더 많이 겹쳐도 함께 남을 수 있습니다. 이 예제의 첫 두 예측은 같은 클래스이고 서로 IoU가 약 0.90이라 NMS 기준 0.50과 0.95에서 다르게 남습니다.

이는 고전 NMS 원리 실습입니다. 모든 탐지 모델의 기본 추론 과정에 별도의 NMS가 있다고 일반화하지 않습니다.

공식 함수 설명: [torchvision batched_nms](https://docs.pytorch.org/vision/stable/generated/torchvision.ops.batched_nms.html), [box_iou](https://docs.pytorch.org/vision/stable/generated/torchvision.ops.box_iou.html).

In [5]:
for nms_iou in [0.50, 0.95]:
    local_keep = batched_nms(boxes, scores, classes, nms_iou)
    print("NMS IoU:", nms_iou, "남긴 인덱스:", local_keep.tolist())


NMS IoU: 0.5 남긴 인덱스: [0, 2, 3]
NMS IoU: 0.95 남긴 인덱스: [0, 1, 2, 3]


## 5. 줄인 배열의 인덱스와 원본 인덱스

함수에 원본 전체가 아닌 `boxes[ids]`를 넘겼다면 반환 인덱스는 **줄인 배열 안의 위치**입니다. 원본 위치가 필요하면 `keep = ids[local_keep]`로 변환합니다.

이 데이터는 confidence 필터 결과가 앞부분에 연속해서 모여 있어 위치 번호가 우연히 같아 보일 수 있습니다. 아래는 `ids=[1,3]`으로 작은 예를 만들어 두 번호의 차이를 드러냅니다. 이 예는 인덱스의 의미를 설명하기 위한 별도 입력입니다.

In [6]:
example_ids = torch.tensor([1, 3])
local_keep = batched_nms(boxes[example_ids], scores[example_ids], classes[example_ids], 0.5)
print("전달한 원본 위치:", example_ids.tolist())
print("반환된 부분 배열 위치:", local_keep.tolist())
print("원본 위치로 변환:", example_ids[local_keep].tolist())

# 실습 A02에서 사용되는 계산 형태
conf, nms_iou = 0.25, 0.5
ids = torch.where(scores >= conf)[0]
local_keep = batched_nms(boxes[ids], scores[ids], classes[ids], nms_iou)
keep = ids[local_keep]
print("이번 실습의 원본 인덱스:", keep.tolist())


전달한 원본 위치: [1, 3]
반환된 부분 배열 위치: [0, 1]
원본 위치로 변환: [1, 3]
이번 실습의 원본 인덱스: [0, 2, 3]


## 6. 정답 매칭: 남은 예측을 TP·FP로 구분하기

- TP: 같은 이미지·같은 클래스의 아직 연결되지 않은 정답과 IoU가 평가 기준 이상인 예측.
- FP: 연결할 정답이 없는 예측. 같은 정답의 중복 예측도 포함됩니다.
- FN: 어떤 예측과도 연결되지 않은 정답.

여기서는 매칭 IoU를 **0.50으로 고정**합니다. `used`는 이미 연결한 정답의 위치를 저장하는 집합(set)입니다. 집합에 있는 정답은 다시 연결하지 않습니다. `eligible`은 클래스·이미지 조건이 맞고 아직 사용하지 않은 정답 후보입니다. `sorted(..., key=...)`는 IoU가 큰 후보부터 살펴보게 합니다.

아래 함수는 학습용 예제이며 원본 파일을 실행하거나 수정하지 않습니다.

In [7]:
def summarize(conf, nms_iou):
    ids = torch.where(scores >= conf)[0]
    keep_local = batched_nms(boxes[ids], scores[ids], classes[ids], nms_iou)
    keep = ids[keep_local]
    used = set()
    details = []
    for i in keep.tolist():
        overlaps = box_iou(boxes[[i]], gt_boxes)[0]
        eligible = [j for j in range(len(g))
                    if int(classes[i]) == int(g.iloc[j]["class_id"])
                    and p.iloc[i]["image_id"] == g.iloc[j]["image_id"]
                    and j not in used]
        matched = None
        for j in sorted(eligible, key=lambda j: -float(overlaps[j])):
            if float(overlaps[j]) >= 0.5:
                used.add(j)
                matched = j
                break
        details.append({"prediction_index": i, "GT_index": matched,
                        "TP": int(matched is not None), "FP": int(matched is None)})
    tp = len(used)
    fp = len(keep) - tp
    fn = len(g) - tp
    summary = dict(conf=conf, nms_iou=nms_iou, kept=len(keep), tp=tp, fp=fp, fn=fn,
                   precision=tp / (tp + fp) if tp + fp else 0,
                   recall=tp / (tp + fn) if tp + fn else 0)
    return summary, pd.DataFrame(details)

summary, details = summarize(0.25, 0.5)
print(details.to_string(index=False))
print(summary)


 prediction_index  GT_index  TP  FP
                0       0.0   1   0
                2       1.0   1   0
                3       NaN   0   1
{'conf': 0.25, 'nms_iou': 0.5, 'kept': 3, 'tp': 2, 'fp': 1, 'fn': 1, 'precision': 0.6666666666666666, 'recall': 0.6666666666666666}


## 7. Precision과 Recall을 숫자로 읽기

**Precision = TP / (TP+FP)**: 남긴 예측 중 정답과 연결된 비율입니다. **Recall = TP / (TP+FN)**: 전체 정답 중 찾은 비율입니다. 분모가 0일 때 이 예제는 0으로 기록합니다.

conf=0.25, NMS IoU=0.50에서는 TP=2, FP=1, FN=1이므로 두 비율 모두 2/3입니다. conf를 0.50으로 높이면 점수 0.40인 예측이 제외됩니다. NMS IoU를 0.95로 높이면 겹친 예측이 더 남습니다. 아래 표에서는 두 기준을 한 번에 모두 바꾸지 않고 각각의 조건을 나란히 봅니다.

AP·mAP는 여러 confidence 순위의 Precision–Recall 관계를 요약한 별도 지표입니다. 이 표는 각 조건 하나에서의 precision·recall입니다.

In [8]:
comparison = pd.DataFrame([summarize(conf, nms_iou)[0]
                           for conf, nms_iou in [(0.25, 0.50), (0.50, 0.50), (0.25, 0.95)]])
print(comparison.round(4).to_string(index=False))


 conf  nms_iou  kept  tp  fp  fn  precision  recall
 0.25     0.50     3   2   1   1     0.6667  0.6667
 0.50     0.50     2   2   0   1     1.0000  0.6667
 0.25     0.95     4   2   2   1     0.5000  0.6667


## 8. 직접 기록하기

아래 항목은 직접 작성하세요.

- 현재 `02_nms.py`의 반환 인덱스가 어느 배열의 위치인지:
- 직접 실행한 명령과 환경:
- 내가 관찰한 수치:
- 검증 여부와 판단 근거:
- 수치만으로 알 수 없는 점:

연습: confidence를 0.90으로 바꿔 남은 예측 수와 recall을 계산해 보세요. 그다음 NMS 기준만 0.80으로 바꿔 두 번째 박스가 남는지 살펴보세요. 원본 데이터 대신 이 노트북의 변수만 바꿔 비교할 수 있습니다.